importing libraries
loads the raw text
splits into chunks
creates document embeddings for chunks
stores embeddings in vector database

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

C:\Users\scorpious\AppData\Local\Temp\ipykernel_14180\3129460803.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


loading the book dataset

In [2]:
import pandas as pd
books = pd.read_csv("books_with_emotions.csv")


  Export descriptions to a text file as langchain doesnot work with pandas

In [3]:
books["tagged_description"].to_csv(
    "tagged_description.txt",    index=False,
    header=False
)

Loading and splitting  the text file into documents

In [4]:
raw_documents = TextLoader("tagged_description.txt", encoding="latin-1").load()
text_splitter = CharacterTextSplitter(chunk_size=2000, chunk_overlap=0, separator="\n")
documents = text_splitter.split_documents(raw_documents)

Created a chunk of size 2016, which is longer than the specified 2000
Created a chunk of size 2012, which is longer than the specified 2000
Created a chunk of size 2834, which is longer than the specified 2000
Created a chunk of size 2510, which is longer than the specified 2000
Created a chunk of size 2014, which is longer than the specified 2000
Created a chunk of size 2291, which is longer than the specified 2000
Created a chunk of size 2616, which is longer than the specified 2000
Created a chunk of size 2046, which is longer than the specified 2000
Created a chunk of size 2762, which is longer than the specified 2000
Created a chunk of size 2005, which is longer than the specified 2000
Created a chunk of size 2155, which is longer than the specified 2000
Created a chunk of size 2209, which is longer than the specified 2000
Created a chunk of size 2877, which is longer than the specified 2000
Created a chunk of size 2400, which is longer than the specified 2000
Created a chunk of s

 Building  the vector database

In [8]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings

db_books = Chroma.from_documents(
    documents,
    embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"),
    persist_directory="chroma_db",      # ← save to disk
    collection_name="books"             # ← name it "books"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [7]:
query = "A book to teach children about nature"
docs = db_books.similarity_search(query, k=10)
docs

[Document(id='05de3a7e-a03e-4d69-962e-2fbe7739b1b1', metadata={'source': 'tagged_description.txt'}, page_content='"9780064404419 ""Culled from 69 stories collected in a [1930s] WPA project, [these 20] tales are organized into sections with themes like \'Tricksters\' or \'Virtues and Vices,\' each with a thoughtful introduction placing the individual stories in the context of feelings and background of the original tellers. Yep\'s telling is vigorous, often poetic, imbued with earthy humor and realism touched with fatalism. A handsomely designed collection."" â\x80\x94K. Notable Children\'s Books of 1989 (ALA) The USA Through Children\'s Books 1990 (ALA) 1989 Boston Globeâ\x80\x93Horn Book Award for Nonfiction 1990 Fanfare Honor List (The Horn Book) 1989 Children\'s Editors\' Choices (BL) Notable 1989 Children\'s Trade Books in Social Studies (NCSS/CBC) Children\'s Books of 1989 (Library of Congress) 1989 Children\'s Books (NY Public Library) ""The Best Books"" 1989 (Parents Magazine)"\

 Matching  a result back to the books table

In [8]:
books[books["isbn13"] == int(docs[0].page_content.strip('"').split()[0].strip())]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
407,9780064404419,0064404412,The Rainbow People,Laurence Yep,Juvenile Fiction,http://books.google.com/books/content?id=5AHwq...,"""Culled from 69 stories collected in a [1930s]...",1992.0,3.75,208.0,202.0,The Rainbow People:,"9780064404419 ""Culled from 69 stories collecte..."


 final recommendation function

In [9]:
def retrieve_semantic_recommendations(
        query: str,
        top_k: int = 10,
) -> pd.DataFrame:
    recs = db_books.similarity_search(query, k=50)

    books_list = []
    for i in range(0, len(recs)):
        books_list += [int(recs[i].page_content.strip('"').split()[0])]

    return books[books["isbn13"].isin(books_list)]

In [7]:
retrieve_semantic_recommendations("A book about fairy tale ")

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,tagged_description,simple_categories,anger,disgust,fear,joy,sadness,surprise,neutral
56,9780007137336,0007137338,Lirael,Garth Nix,Fantasy fiction,http://books.google.com/books/content?id=sDzU8...,When a dangerous necromancer threatens to unle...,2004.0,4.30,527.0,1339.0,9780007137336 When a dangerous necromancer thr...,Nonfiction,0.064134,0.104007,0.957042,0.040564,0.942740,0.111690,0.078766
146,9780060556501,0060556501,"The Lion, the Witch and the Wardrobe (picture ...",C. S. Lewis,Juvenile Fiction,http://books.google.com/books/content?id=FlSpp...,Narnia: A magical land full of wonder and exci...,2004.0,4.19,48.0,5012.0,9780060556501 Narnia: A magical land full of w...,Children's Fiction,0.500388,0.146838,0.247717,0.863717,0.944420,0.494298,0.621472
216,9780060760823,0060760826,A Couple of April Fools,Gregory Maguire,Juvenile Fiction,http://books.google.com/books/content?id=qW4cv...,"At a Vermont elementary school, April Fools' D...",2005.0,3.50,240.0,3.0,"9780060760823 At a Vermont elementary school, ...",Children's Fiction,0.064134,0.601675,0.319988,0.040564,0.549477,0.111690,0.078766
340,9780060987527,0060987529,Confessions of an Ugly Stepsister,Gregory Maguire,Fiction,http://books.google.com/books/content?id=R7lwa...,Is this new land a place where magics really h...,2000.0,3.52,372.0,53017.0,9780060987527 Is this new land a place where m...,Fiction,0.477070,0.712513,0.823898,0.270591,0.781858,0.308516,0.429871
378,9780061139376,0061139378,Coraline,Neil Gaiman,Fiction,http://books.google.com/books/content?id=pXlAY...,In Coraline's family's new flat there's a lock...,2006.0,4.05,162.0,377685.0,9780061139376 In Coraline's family's new flat ...,Fiction,0.393250,0.179143,0.938569,0.378467,0.874423,0.111690,0.488982
440,9780066238500,0066238501,The Chronicles of Narnia (adult),C. S. Lewis,Fiction,http://books.google.com/books/content?id=3VGkK...,"Journeys to the end of the world, fantastic cr...",2001.0,4.26,767.0,425445.0,9780066238500 Journeys to the end of the world...,Fiction,0.064134,0.104007,0.051363,0.487125,0.951270,0.111690,0.130764
558,9780140071788,0140071784,Roald Dahl's Book of Ghost Stories,Roald Dahl,"Ghost stories, American",http://books.google.com/books/content?id=zGt6R...,Who better to investigate the literary spirit ...,1985.0,3.69,249.0,2403.0,9780140071788 Who better to investigate the li...,Fiction,0.064134,0.104007,0.773546,0.040564,0.549477,0.253176,0.078766
646,9780140367430,0140367438,The Enchanted Castle,E. Nesbit,Juvenile Fiction,http://books.google.com/books/content?id=75V6x...,"E. Nesbit's classic story of how Gerald, Cathy...",2004.0,3.85,291.0,6124.0,9780140367430 E. Nesbit's classic story of how...,Children's Fiction,0.090869,0.165256,0.749953,0.120340,0.549477,0.111690,0.768096
648,9780140430363,0140430369,Three Gothic Novels,Horace Walpole;William Beckford;Mary Shelley,Fiction,http://books.google.com/books/content?id=ZjI_D...,"Walpole's The Castle of Otranto, Beckford's Va...",1968.0,3.70,505.0,290.0,"9780140430363 Walpole's The Castle of Otranto,...",Fiction,0.064134,0.104007,0.982916,0.698616,0.877477,0.111690,0.163506
682,9780140445503,0140445501,Gargantua and Pantagruel,Francois Rabelais;Michael Andrew Screech,Biography & Autobiography,http://books.google.com/books/content?id=fWwts...,The dazzling and exuberant moral stories of Ra...,2006.0,3.71,1041.0,11605.0,9780140445503 The dazzling and exuberant moral...,Nonfiction,0.118239,0.645261,0.051363,0.961089,0.549477,0.111690,0.078766
